# Ensemble 8-de-8 — Validacao em Dataset Sintetico Conhecido (Toy A / Toy B)

Este notebook roda a mesma suite de 8 algoritmos de descoberta causal registrados no framework
(`ClassicalGranger, GES, FCI, DYNOTEARS, LPCMCI, NeuralGrangercMLP, PCMCI, VARLiNGAM`) e a
combinacao por ensemble (soft voting ponderado) contra dois datasets sinteticos com grafo causal
totalmente conhecido: `toy_a_linear` (relacoes lineares) e `toy_b_nonlinear` (relacoes via
`tanh`). Ambos tem 5 variaveis (`Y, X1, X3, X4, X0`) e exatamente 3 arestas causais diretas:
`X1->Y`, `X3->Y`, `X4->X1` — `X0` e ruido puro e `X4->Y` e somente indireto (via X1).

**Este nao e o mesmo experimento do `Benchmark_Superioridade_Ensemble_Traffic.ipynb`.** Aquele
notebook compara o ensemble contra cada metodo individual em 10 trajetorias pareadas do mesmo
grafo Traffic, com um protocolo estatistico confirmatorio (Wilcoxon + Holm, IC 95%, taxa de
vitoria). Aqui ha **uma unica serie continua de 20000 passos por dataset** — nao existem
trajetorias para parear, entao nenhuma alegacao estatistica confirmatoria e feita aqui.

**Proposito:** validacao de sanidade/demonstracao — checar se o ensemble (e cada metodo
individual) recupera corretamente as 3 arestas conhecidas, sem incluir X0 e sem confundir a
relacao indireta X4->Y com uma aresta direta. Ajustar parametros aqui e diagnostico: mostra que o
framework funciona num caso totalmente controlado, mas nao substitui nem generaliza o protocolo
confirmatorio do notebook Traffic.

In [1]:
from pathlib import Path
import json
import os
import pickle
import time

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

from causal_discovery import (
    CausalPreprocessor,
    add_precision_consensus_selection,
    build_complete_undirected_pair_scores,
    compute_ranked_undirected_skeleton_metrics,
    compute_undirected_skeleton_metrics,
    get_registered_method_kwargs,
    get_registered_method_weights,
    get_registered_methods,
    load_time_series_dataset,
)
from causal_discovery.ensemble_selection import (
    add_ranked_structure_selection,
    select_robust_ensemble_combination,
)

## 1. Configuracao

Os oito metodos registrados, `MAX_LAG = 1` (o DGP verdadeiro e lag-1 em todas as arestas —
compativel com os 8 metodos; PCMCI e NeuralGrangercMLP tratam `max_lag=1` como piso minimo
suportado, os demais nao tem piso). `N_BOOTSTRAP = 10` na primeira passada (nao 30 como no
Traffic): cada bootstrap reexecuta a suite inteira de 8 metodos sobre as ~20000 linhas, e o
NeuralGrangercMLP retreina uma rede do zero a cada chamada — bem mais caro que as trajetorias de
40 linhas do Traffic. `max_bootstrap_seconds` limita o tempo total da fase de bootstrap por
dataset. Os limiares de selecao de pares foram recalibrados para 5 nos (10 pares candidatos, 3
verdadeiros ⇒ prevalencia 30%) em vez de herdados do ajuste feito para 10 nos/45 pares do Traffic.

In [2]:
DATASETS = {
    "toy_a_linear": {
        "data_path": Path("datasets/synthetic_causal/toy_a_linear.csv"),
        "ground_truth_path": Path("datasets/synthetic_causal/toy_a_linear_gt.csv"),
    },
    "toy_b_nonlinear": {
        "data_path": Path("datasets/synthetic_causal/toy_b_nonlinear.csv"),
        "ground_truth_path": Path("datasets/synthetic_causal/toy_b_nonlinear_gt.csv"),
    },
}

RESULTS_DIR = Path(".local/results/toy_synthetic_validation")
CACHE_DIR = Path(".local/results/toy_synthetic_validation_cache")

ENSEMBLE_METHOD_NAMES = [
    "ClassicalGranger", "GES", "FCI", "DYNOTEARS",
    "LPCMCI", "NeuralGrangercMLP", "PCMCI", "VARLiNGAM",
]
MAX_LAG = 1
ENSEMBLE_THRESHOLD = 0.50
N_BOOTSTRAP = 10
METHOD_CONSENSUS_THRESHOLD = 0.50
SOFT_VOTING_SUPPORT_THRESHOLD = 0.0
LOCAL_EXPERT_WEIGHT = 0.60
PREDICTIVE_VALIDATION_WEIGHT = 0.75
PREDICTIVE_VALIDATION_SPLITS = 3
PREDICTIVE_VALIDATION_RIDGE_ALPHA = 5.0
PREDICTIVE_VALIDATION_CONDITIONAL_PARENTS = 0
PREDICTIVE_VALIDATION_UNCERTAINTY_PENALTY = 0.50
PREDICTIVE_VALIDATION_RANK_EXPONENT = 0.35
RANKED_SELECTION_MAX_PAIR_DENSITY = 0.30
RANKED_SELECTION_RESCUE_BOOTSTRAP_MIN = 0.40
RANKED_SELECTION_RESCUE_PREDICTIVE_RANK_MIN = 0.60
RANKED_SELECTION_RESCUE_SUPPORT_MIN = 0.25
METHOD_REDUNDANCY_PENALTY = 0.20
RANDOM_STATE = 42
MAX_BOOTSTRAP_SECONDS = 900

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Metodos do ensemble: {ENSEMBLE_METHOD_NAMES}")
print(f"MAX_LAG={MAX_LAG} | N_BOOTSTRAP={N_BOOTSTRAP} | RANDOM_STATE={RANDOM_STATE}")

Metodos do ensemble: ['ClassicalGranger', 'GES', 'FCI', 'DYNOTEARS', 'LPCMCI', 'NeuralGrangercMLP', 'PCMCI', 'VARLiNGAM']
MAX_LAG=1 | N_BOOTSTRAP=10 | RANDOM_STATE=42


## 2. Carregamento e pre-processamento

Cada dataset e carregado com o loader de CSV estendido (`ground_truth_path`), que interpreta o
CSV `Edge,Direct,Coefficient,Lag,Type` gerado por `synthetic_causal_datasets.ipynb` e mantem
apenas as arestas com `Direct=True` como ground truth. O pre-processamento usa o mesmo
`CausalPreprocessor` do notebook Traffic (teste ADF + diferenciacao condicional + normalizacao).
Como os coeficientes AR do gerador (<= 0.7) estao bem abaixo da raiz unitaria, a serie ja deve ser
estacionaria — a ordem de diferenciacao de cada variavel e impressa abaixo para confirmar que
nenhuma diferenciacao extra distorce a estrutura de lag=1 que queremos recuperar.

In [3]:
def load_dataset_bundle(name):
    config = DATASETS[name]
    return load_time_series_dataset(
        config["data_path"],
        data_format="csv",
        ground_truth_path=config["ground_truth_path"],
        selected_columns=None,
    )


def preprocess_series(raw):
    preprocessor = CausalPreprocessor(
        raw, significance_level=0.05, decomposition_period=None
    )
    processed = preprocessor.fit_transform(
        make_stationary=True, normalize=True, remove_trend=False, max_diffs=2,
    )
    return processed, preprocessor


bundles = {name: load_dataset_bundle(name) for name in DATASETS}
truth_summaries = {}

for name, bundle in bundles.items():
    nodes = list(bundle.selected_columns)
    truth_summary = compute_undirected_skeleton_metrics(
        pd.DataFrame(columns=["source", "target", "lag"]),
        bundle.ground_truth,
        nodes=nodes,
    )
    truth_summaries[name] = truth_summary
    print(f"--- {name} ---")
    print(f"Linhas: {len(bundle.data)} | Nos: {nodes}")
    print(f"Pares possiveis: {truth_summary['candidate_pairs']} | "
          f"Pares verdadeiros: {truth_summary['ground_truth_pairs']} | "
          f"Prevalencia: {truth_summary['ground_truth_prevalence']:.2%}")
    print(f"Arestas do ground truth: "
          f"{list(zip(bundle.ground_truth['source'], bundle.ground_truth['target']))}")

processed_series = {}
for name, bundle in bundles.items():
    processed, preprocessor = preprocess_series(bundle.data)
    print(f"{name}: linhas apos pre-processamento={len(processed)} | "
          f"ordens de diferenciacao={preprocessor.differencing_orders}")
    processed_series[name] = processed

--- toy_a_linear ---
Linhas: 20000 | Nos: ['Y', 'X1', 'X3', 'X4', 'X0']
Pares possiveis: 10 | Pares verdadeiros: 3 | Prevalencia: 30.00%
Arestas do ground truth: [('X1', 'Y'), ('X3', 'Y'), ('X4', 'X1')]
--- toy_b_nonlinear ---
Linhas: 20000 | Nos: ['Y', 'X1', 'X3', 'X4', 'X0']
Pares possiveis: 10 | Pares verdadeiros: 3 | Prevalencia: 30.00%
Arestas do ground truth: [('X1', 'Y'), ('X3', 'Y'), ('X4', 'X1')]


toy_a_linear: linhas apos pre-processamento=20000 | ordens de diferenciacao={'Y': 0, 'X1': 0, 'X3': 0, 'X4': 0, 'X0': 0}


toy_b_nonlinear: linhas apos pre-processamento=20000 | ordens de diferenciacao={'Y': 0, 'X1': 0, 'X3': 0, 'X4': 0, 'X0': 0}


## 3. Funcoes do experimento (adaptadas do notebook Traffic)

Mesmas funcoes de avaliacao/agregacao do notebook Traffic (`restrict_method_relations`,
`evaluate_strategy`, `baseline_rows`), parametrizadas por `nodes`/`ground_truth` explicitos em vez
de globais, ja que aqui rodamos dois datasets na mesma sessao. O gate preditivo e a selecao
ranqueada seguem exatamente a mesma logica: score de ensemble ponderado, corte suave em `0,50`, e
um resgate por evidencia preditiva complementar para pares fora do teto de densidade.

In [4]:
def restrict_method_relations(method, allowed_relations):
    allowed_relations = set(allowed_relations)

    def run_restricted(data, **kwargs):
        result = method(data, **kwargs)
        if result is None or result.empty:
            return result
        mask = [
            (source, target) in allowed_relations
            for source, target in zip(result["source"], result["target"])
        ]
        return result.loc[mask].reset_index(drop=True)

    return run_restricted


def evidence_mode(frame):
    if "p_value" in frame.columns:
        p_values = pd.to_numeric(frame["p_value"], errors="coerce")
        if np.isfinite(p_values).any():
            return "one_minus_p_value"
    return "absolute_score"


def apply_precision_focused_consensus(summary):
    return add_precision_consensus_selection(
        summary,
        score_threshold=ENSEMBLE_THRESHOLD,
        method_support_threshold=METHOD_CONSENSUS_THRESHOLD,
    )


def apply_soft_voting_precision(summary):
    frame = summary.copy()
    frame["ensemble_score"] = frame["pre_validation_ensemble_score"]
    return add_precision_consensus_selection(
        frame,
        score_threshold=ENSEMBLE_THRESHOLD,
        method_support_threshold=SOFT_VOTING_SUPPORT_THRESHOLD,
    )


def evaluate_strategy(frame, strategy, nodes, ground_truth, runtime_seconds, *, probability=False):
    binary_frame = frame
    binary_threshold = ENSEMBLE_THRESHOLD
    if probability and "ensemble_selected" in frame:
        binary_frame = frame.loc[
            frame["ensemble_selected"].fillna(False).astype(bool)
        ].copy()
        binary_threshold = 0.0
    binary = compute_undirected_skeleton_metrics(
        binary_frame, ground_truth, prob_threshold=binary_threshold, nodes=nodes,
    )
    pair_scores = build_complete_undirected_pair_scores(
        frame, nodes,
        evidence=(
            "ensemble_score" if probability and "ensemble_score" in frame.columns
            else "probability" if probability
            else evidence_mode(frame)
        ),
    )
    ranked = compute_ranked_undirected_skeleton_metrics(pair_scores, ground_truth)
    return {
        "strategy": str(strategy),
        "precision": binary["precision"],
        "recall": binary["recall"],
        "f1_score": binary["f1_score"],
        "structural_hamming_distance": binary["structural_hamming_distance"],
        "true_positives": binary["true_positives"],
        "false_positives": binary["false_positives"],
        "false_negatives": binary["false_negatives"],
        "average_precision": ranked["average_precision"],
        "roc_auc": ranked["roc_auc"],
        "runtime_seconds": float(runtime_seconds),
    }


def baseline_rows(nodes, ground_truth, truth_summary):
    pairs = [(nodes[i], nodes[j]) for i in range(len(nodes)) for j in range(i + 1, len(nodes))]
    all_pairs = pd.DataFrame([
        {"source": source, "target": target, "lag": 1, "score": 1.0, "p_value": np.nan}
        for source, target in pairs
    ])
    rng = np.random.default_rng(RANDOM_STATE)
    random_scores = rng.random(len(pairs))
    true_pair_count = truth_summary["ground_truth_pairs"]
    selected = (
        np.argsort(random_scores)[-true_pair_count:]
        if true_pair_count else np.array([], dtype=int)
    )
    random_edges = pd.DataFrame([
        {"source": pairs[index][0], "target": pairs[index][1], "lag": 1,
         "score": random_scores[index], "p_value": np.nan}
        for index in selected
    ])
    random_pair_scores = pd.DataFrame([
        {"source": source, "target": target, "score": score}
        for (source, target), score in zip(pairs, random_scores)
    ])

    all_metrics = evaluate_strategy(all_pairs, "ALL_PAIRS", nodes, ground_truth, 0.0)
    random_binary = compute_undirected_skeleton_metrics(random_edges, ground_truth, nodes=nodes)
    random_ranked = compute_ranked_undirected_skeleton_metrics(random_pair_scores, ground_truth)
    random_metrics = {
        "strategy": "RANDOM_DENSITY",
        "precision": random_binary["precision"], "recall": random_binary["recall"],
        "f1_score": random_binary["f1_score"],
        "structural_hamming_distance": random_binary["structural_hamming_distance"],
        "true_positives": random_binary["true_positives"],
        "false_positives": random_binary["false_positives"],
        "false_negatives": random_binary["false_negatives"],
        "average_precision": random_ranked["average_precision"],
        "roc_auc": random_ranked["roc_auc"], "runtime_seconds": 0.0,
    }
    return [all_metrics, random_metrics]


def selection_arguments(processed_data):
    return {
        "min_methods": len(ENSEMBLE_METHOD_NAMES),
        "max_methods": len(ENSEMBLE_METHOD_NAMES),
        "min_votes": 1,
        "n_bootstrap": N_BOOTSTRAP,
        "block_size": max(2, len(processed_data) // 12),
        "stability_threshold": 0.60,
        "selection_probability_threshold": 0.50,
        "prior_edge_probability": 0.10,
        "posterior_weight": 0.70,
        "adaptive_method_weights": True,
        "stability_weight": 0.65,
        "local_expert_weight": LOCAL_EXPERT_WEIGHT,
        "predictive_validation_weight": PREDICTIVE_VALIDATION_WEIGHT,
        "predictive_validation_max_lag": MAX_LAG,
        "predictive_validation_splits": PREDICTIVE_VALIDATION_SPLITS,
        "predictive_validation_ridge_alpha": PREDICTIVE_VALIDATION_RIDGE_ALPHA,
        "predictive_validation_conditional_parents": PREDICTIVE_VALIDATION_CONDITIONAL_PARENTS,
        "predictive_validation_uncertainty_penalty": PREDICTIVE_VALIDATION_UNCERTAINTY_PENALTY,
        "predictive_validation_rank_exponent": PREDICTIVE_VALIDATION_RANK_EXPONENT,
        "ranked_selection_max_pair_density": RANKED_SELECTION_MAX_PAIR_DENSITY,
        "ranked_selection_rescue_bootstrap_min": RANKED_SELECTION_RESCUE_BOOTSTRAP_MIN,
        "ranked_selection_rescue_predictive_rank_min": RANKED_SELECTION_RESCUE_PREDICTIVE_RANK_MIN,
        "ranked_selection_rescue_support_min": RANKED_SELECTION_RESCUE_SUPPORT_MIN,
        "method_redundancy_penalty": METHOD_REDUNDANCY_PENALTY,
        "method_stability_power": 1.0,
        "method_diversity_bonus": 0.15,
        "method_density_penalty": 0.50,
        "minimum_method_weight": 0.05,
        "confidence_level": 0.95,
        "random_state": RANDOM_STATE,
        "precompute_runs": True,
        "parallel_jobs": max(1, min(4, (os.cpu_count() or 2) - 1)),
        "max_bootstrap_seconds": MAX_BOOTSTRAP_SECONDS,
    }

## 4. Execucao por dataset (com cache)

Cada dataset roda uma vez a combinacao fixa 8-de-8 (`min_methods=max_methods=8`, sem busca de
subconjuntos). As saidas base e as saidas de cada bootstrap ficam em cache por pickle em
`.local/results/toy_synthetic_validation_cache/` — reexecutar esta celula depois de mudar apenas
parametros da camada de ensemble (limiares, pesos, penalizacoes) reaproveita o cache e nao
reexecuta os 8 metodos.

In [5]:
all_registered_methods = get_registered_methods()
all_method_kwargs = get_registered_method_kwargs(MAX_LAG)
all_method_weights = get_registered_method_weights()
methods = {name: all_registered_methods[name] for name in ENSEMBLE_METHOD_NAMES}
method_kwargs = {name: all_method_kwargs[name] for name in ENSEMBLE_METHOD_NAMES}
method_weights = {name: all_method_weights[name] for name in ENSEMBLE_METHOD_NAMES}


def run_dataset(name, processed, nodes, ground_truth, truth_summary):
    relations = {
        (source, target) for source in nodes for target in nodes if source != target
    }
    restricted = {
        method_name: restrict_method_relations(method, relations)
        for method_name, method in methods.items()
    }

    cache_path = CACHE_DIR / f"{name}.pkl"
    cache_signature = {
        "dataset": name, "nodes": list(nodes), "processed_rows": len(processed),
        "max_lag": MAX_LAG, "n_bootstrap": N_BOOTSTRAP,
        "ensemble_methods": list(ENSEMBLE_METHOD_NAMES),
        "local_expert_weight": LOCAL_EXPERT_WEIGHT,
        "method_redundancy_penalty": METHOD_REDUNDANCY_PENALTY,
        "random_state": RANDOM_STATE,
    }
    cached_payload = None
    if cache_path.exists():
        with cache_path.open("rb") as stream:
            candidate_payload = pickle.load(stream)
        if candidate_payload.get("signature") == cache_signature:
            cached_payload = candidate_payload

    started = time.perf_counter()
    selection = select_robust_ensemble_combination(
        processed, restricted,
        method_kwargs=method_kwargs, method_weights=method_weights,
        expert_knowledge=[],
        precomputed_outputs=(cached_payload.get("outputs") if cached_payload else None),
        precomputed_bootstrap_outputs=(
            cached_payload.get("bootstrap_outputs") if cached_payload else None
        ),
        **selection_arguments(processed),
    )
    recalculation_runtime = time.perf_counter() - started
    cached_base_runtime = cached_payload.get("base_runtime_seconds") if cached_payload else None
    ensemble_runtime = (
        float(cached_base_runtime)
        if cached_base_runtime is not None and np.isfinite(cached_base_runtime)
        else recalculation_runtime
    )
    if cached_payload is None:
        with cache_path.open("wb") as stream:
            pickle.dump({
                "signature": cache_signature,
                "outputs": selection["precomputed_outputs"],
                "bootstrap_outputs": selection["precomputed_bootstrap_outputs"],
                "base_runtime_seconds": recalculation_runtime,
            }, stream, protocol=pickle.HIGHEST_PROTOCOL)

    individual_outputs = {}
    for evaluation in selection["all_evaluations"].values():
        for method_name, output in evaluation["outputs"].items():
            individual_outputs.setdefault(method_name, output)

    rows = [
        evaluate_strategy(
            individual_outputs[method_name], method_name, nodes, ground_truth, np.nan
        )
        for method_name in ENSEMBLE_METHOD_NAMES
    ]

    summary_before_consensus = selection["best_evaluation"]["probabilistic_summary"].copy()
    summary_hard_consensus = apply_precision_focused_consensus(summary_before_consensus)
    summary = apply_soft_voting_precision(summary_before_consensus)
    summary_without_gate = summary_before_consensus.copy()
    summary_without_gate["ensemble_score"] = summary_without_gate["pre_validation_ensemble_score"]
    summary_without_gate = add_ranked_structure_selection(
        summary_without_gate, nodes=nodes,
        max_pair_density=RANKED_SELECTION_MAX_PAIR_DENSITY,
    )

    rows.append(evaluate_strategy(
        summary_hard_consensus, "ENSEMBLE_MAIORIA_RIGIDA", nodes, ground_truth,
        ensemble_runtime, probability=True,
    ))
    rows.append(evaluate_strategy(
        summary_without_gate, "ENSEMBLE_SEM_GATE", nodes, ground_truth,
        ensemble_runtime, probability=True,
    ))
    rows.append(evaluate_strategy(
        summary_before_consensus, "ENSEMBLE_TOP_K_ANTERIOR", nodes, ground_truth,
        ensemble_runtime, probability=True,
    ))
    rows.append(evaluate_strategy(
        summary, "ENSEMBLE", nodes, ground_truth, ensemble_runtime, probability=True,
    ))
    rows.extend(baseline_rows(nodes, ground_truth, truth_summary))
    for row in rows:
        row["dataset"] = name

    weight_diagnostics = selection["best_evaluation"]["method_weight_diagnostics"].set_index("method")
    selected_pair_count = len({
        tuple(sorted((str(row.source), str(row.target))))
        for row in summary.loc[summary["ensemble_selected"]].itertuples()
    })
    selection_row = {
        "dataset": name,
        "best_combination": " + ".join(selection["best_combination"]),
        "ensemble_runtime_seconds": ensemble_runtime,
        "cache_reused": cached_payload is not None,
        "recalculation_runtime_seconds": recalculation_runtime,
        "processed_rows": len(processed),
        "bootstrap_iterations_used": len(selection["precomputed_bootstrap_outputs"]),
        "selected_pair_count": selected_pair_count,
        "adaptive_weights": json.dumps(
            selection["best_evaluation"]["effective_method_weights"],
            ensure_ascii=False, sort_keys=True,
        ),
        "method_redundancy": json.dumps(
            weight_diagnostics["redundancy"].dropna().to_dict(),
            ensure_ascii=False, sort_keys=True,
        ),
    }
    return rows, selection_row, summary


all_rows = []
all_selection_rows = []
summaries = {}

for name, bundle in bundles.items():
    processed = processed_series[name]
    nodes = list(bundle.selected_columns)
    ground_truth = bundle.ground_truth
    truth_summary = truth_summaries[name]
    print(f"Executando ensemble para {name}...")
    started = time.perf_counter()
    rows, selection_row, summary = run_dataset(
        name, processed, nodes, ground_truth, truth_summary
    )
    print(f"{name}: concluido em {time.perf_counter() - started:.1f}s")
    all_rows.extend(rows)
    all_selection_rows.append(selection_row)
    summaries[name] = summary

metrics = pd.DataFrame(all_rows)
selections = pd.DataFrame(all_selection_rows)
print("Execucao concluida.")

Executando ensemble para toy_a_linear...


toy_a_linear: concluido em 86.8s
Executando ensemble para toy_b_nonlinear...


toy_b_nonlinear: concluido em 79.2s
Execucao concluida.


## 5. Resultados

Salva `metrics.csv`, `selections.csv` e `configuration.json` em
`.local/results/toy_synthetic_validation/`, seguindo a mesma convencao de arquivos do notebook
Traffic. A tabela principal mostra ENSEMBLE + os 8 metodos individuais; a tabela completa inclui
as ablacoes (`ENSEMBLE_MAIORIA_RIGIDA`, `ENSEMBLE_SEM_GATE`, `ENSEMBLE_TOP_K_ANTERIOR`) e os
baselines (`ALL_PAIRS`, `RANDOM_DENSITY`) para diagnostico.

In [6]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
metrics.to_csv(RESULTS_DIR / "metrics.csv", index=False)
selections.to_csv(RESULTS_DIR / "selections.csv", index=False)
(RESULTS_DIR / "configuration.json").write_text(
    json.dumps({
        "max_lag": MAX_LAG, "n_bootstrap": N_BOOTSTRAP,
        "ensemble_threshold": ENSEMBLE_THRESHOLD,
        "method_consensus_threshold": METHOD_CONSENSUS_THRESHOLD,
        "soft_voting_support_threshold": SOFT_VOTING_SUPPORT_THRESHOLD,
        "local_expert_weight": LOCAL_EXPERT_WEIGHT,
        "predictive_validation_weight": PREDICTIVE_VALIDATION_WEIGHT,
        "ranked_selection_max_pair_density": RANKED_SELECTION_MAX_PAIR_DENSITY,
        "method_redundancy_penalty": METHOD_REDUNDANCY_PENALTY,
        "ensemble_methods": ENSEMBLE_METHOD_NAMES,
        "random_state": RANDOM_STATE,
        "datasets": {name: str(config["data_path"]) for name, config in DATASETS.items()},
    }, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(f"Resultados salvos em {RESULTS_DIR}/")

main_strategies = [*ENSEMBLE_METHOD_NAMES, "ENSEMBLE"]
main_display = metrics[metrics["strategy"].isin(main_strategies)].copy()
main_display["strategy"] = pd.Categorical(
    main_display["strategy"], categories=["ENSEMBLE", *ENSEMBLE_METHOD_NAMES], ordered=True,
)
main_display = main_display.sort_values(["dataset", "strategy"])
print("\nComparacao principal (ENSEMBLE vs metodos individuais):")
display(main_display[[
    "dataset", "strategy", "precision", "recall", "f1_score",
    "structural_hamming_distance", "true_positives", "false_positives",
    "false_negatives", "average_precision", "roc_auc",
]])

print("\nTabela completa (ablacoes + baselines):")
display(metrics.sort_values(["dataset", "strategy"])[[
    "dataset", "strategy", "precision", "recall", "f1_score", "average_precision", "roc_auc",
]])

Resultados salvos em .local\results\toy_synthetic_validation/

Comparacao principal (ENSEMBLE vs metodos individuais):


,dataset,strategy,precision,recall,f1_score,structural_hamming_distance,true_positives,false_positives,false_negatives,average_precision,roc_auc
11,toy_a_linear,ENSEMBLE,1.00,1.000000,1.000000,0,3,0,0,1.000000,1.000000
0,toy_a_linear,ClassicalGranger,0.75,1.000000,0.857143,1,3,1,0,0.750000,0.928571
1,toy_a_linear,GES,1.00,1.000000,1.000000,0,3,0,0,1.000000,1.000000
2,toy_a_linear,FCI,1.00,1.000000,1.000000,0,3,0,0,1.000000,1.000000
3,toy_a_linear,DYNOTEARS,1.00,1.000000,1.000000,0,3,0,0,1.000000,1.000000
4,toy_a_linear,LPCMCI,0.00,0.000000,0.000000,3,0,0,3,0.300000,0.500000
5,toy_a_linear,NeuralGrangercMLP,0.30,1.000000,0.461538,7,3,7,0,0.766667,0.666667
6,toy_a_linear,PCMCI,0.75,1.000000,0.857143,1,3,1,0,1.000000,1.000000
7,toy_a_linear,VARLiNGAM,1.00,1.000000,1.000000,0,3,0,0,1.000000,1.000000
25,toy_b_nonlinear,ENSEMBLE,1.00,1.000000,1.000000,0,3,0,0,1.000000,1.000000



Tabela completa (ablacoes + baselines):


,dataset,strategy,precision,recall,f1_score,average_precision,roc_auc
12,toy_a_linear,ALL_PAIRS,0.300000,1.000000,0.461538,0.300000,0.500000
0,toy_a_linear,ClassicalGranger,0.750000,1.000000,0.857143,0.750000,0.928571
3,toy_a_linear,DYNOTEARS,1.000000,1.000000,1.000000,1.000000,1.000000
11,toy_a_linear,ENSEMBLE,1.000000,1.000000,1.000000,1.000000,1.000000
8,toy_a_linear,ENSEMBLE_MAIORIA_RIGIDA,1.000000,1.000000,1.000000,1.000000,1.000000
9,toy_a_linear,ENSEMBLE_SEM_GATE,1.000000,1.000000,1.000000,1.000000,1.000000
10,toy_a_linear,ENSEMBLE_TOP_K_ANTERIOR,1.000000,1.000000,1.000000,1.000000,1.000000
2,toy_a_linear,FCI,1.000000,1.000000,1.000000,1.000000,1.000000
1,toy_a_linear,GES,1.000000,1.000000,1.000000,1.000000,1.000000
4,toy_a_linear,LPCMCI,0.000000,0.000000,0.000000,0.300000,0.500000


In [7]:
chart_strategies = [*ENSEMBLE_METHOD_NAMES, "ENSEMBLE", "ALL_PAIRS", "RANDOM_DENSITY"]
chart_data = metrics[metrics["strategy"].isin(chart_strategies)]
fig = px.bar(
    chart_data, x="strategy", y="precision", color="dataset", barmode="group",
    category_orders={"strategy": ["ENSEMBLE", *ENSEMBLE_METHOD_NAMES, "ALL_PAIRS", "RANDOM_DENSITY"]},
    title="Precisao por estrategia — toy_a_linear vs toy_b_nonlinear",
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

## 6. Conclusao e limites

Esta execucao e uma validacao de sanidade/demonstracao, **nao** um teste estatistico
confirmatorio: cada dataset tem uma unica serie (nao ha trajetorias pareadas para aplicar
Wilcoxon/Holm/taxa de vitoria como no notebook Traffic). O objetivo era verificar se o ensemble e
os 8 metodos individuais recuperam corretamente as 3 arestas conhecidas (`X1->Y`, `X3->Y`,
`X4->X1`), sem incluir `X0` (ruido puro) e sem confundir a relacao indireta `X4->Y` com uma aresta
direta.

Ajustar os parametros de selecao do ensemble (limiares, penalizacao de redundancia, densidade de
pares) contra esse ground truth e diagnostico: mostra que o framework funciona corretamente num
caso totalmente controlado, mas **nao** demonstra superioridade generalizavel — essa alegacao
continua restrita ao protocolo confirmatorio do notebook Traffic, e mesmo la, restrita ao grafo
Traffic.